<a href="https://colab.research.google.com/github/Onitsiky/MBD-Assignment-3/blob/main/MBBM_Assignmet3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# installing required packages
!pip install openai
!pip install tqdm
!pip install fire

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.4 MB/s eta 0:00:00


In [ ]:
!pip install openai

In [4]:

import pandas as pd
import numpy as np
from tqdm import tqdm
#displays
from IPython.display import display
# limit to the lenght of token that should be displayed on the daatsets
pd.options.display.max_colwidth = 500
from google.colab import userdata
import os

from openai import OpenAI


In [118]:
# getting key

os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
CMU_API_KEY = userdata.get("CMU_API_KEY")

In [6]:
# downloading data from kaggle
import kaggle

!kaggle datasets download -d adityasureshgithub/digital-extremism-detection-curated-dataset
!unzip digital-extremism-detection-curated-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/adityasureshgithub/digital-extremism-detection-curated-dataset
License(s): Community Data License Agreement - Permissive - Version 1.0
100% 118k/118k [00:00<00:00, 63.2MB/s]

Archive:  digital-extremism-detection-curated-dataset.zip
  inflating: extremism_data_final.csv  


In [7]:
# opening the downloaded df
test_df = pd.read_csv('/content/extremism_data_final.csv')
test_df.head()

,Original_Message,Extremism_Label
0,sixth forms should burn to the ground,EXTREMIST
1,whatever should burn benders to the ground,EXTREMIST
2,factories should burn to the ground,EXTREMIST
3,halal factories should burn to the ground,EXTREMIST
4,nhs should burn to the ground,EXTREMIST


# A. Enhance your previous extremism detection system

**Based on the reading, According to the author what are some key differences and similarities between VE and NVE. **<br>
```text
The similarities associated with both VE and NVE include
- Dehumanization is common to both VE and NVE though VE felt a personal responsibility to act compared to NVE and it does not always lead to physical violence
- The study found that both groups exhibit a strong psychological need for sense of purpose and that it does not specifically drive violent behavior.
- Both groups used the internet to connect with like-minded individuals where over 80% were connected with others who shared similar radical views.
- Psychological issues and trauma showed no difference between the two groups e.g.., loss of significant other was a trauma experienced by roughly half of both VE’s and NVE’s. They were  both in search of identity and desire to belong
- The individuals (NVE and VE) both held the extremist ideology regardless of whether they were violent or not
```
<br>

```text
Differences associated with both NVE and VE
- VE were more likely to have been exposed to extreme violence and trauma compared to NVE’s which include internet materials
- Negative social experiences such as being bullied during adolescence is more common to VE compared to NVE’s
- VE often experienced low self esteem and sense of underachievement in their academic achievements or employment status.
- VE felt strongly the need to act based on their belief compared to NVE’s where they tend to operate in an open environment where there are fewer constraints.
- VE are likely to have attended or travelled abroad to train and meet with others who were more influential. They also were likely to have participated in sports earlier in life.
```

**Based on the paper's method section, how are radicalization and extremism defined**<br>
```text
Most materials propose radicalization as a process whereby a person’s beliefs become more extreme where they define it as the process by which people come to support terrorism and extremism and, in some cases, participate in terrorist activity.
It is also highlighted that though it is perceived as a grounding step for extremism, it is not always the case as it does not always result in violent action.
For extremism, the author describes based on UK Government definition which terms it as the vocal or active opposition of fundamental British values which include democracy, rule of law, individual liberty and mutual respect and tolerance of different faiths and beliefs.
The authors for the research categorize extremists as individuals who hold opposing/contradictory attitudes and opinions from the mainstream regarding political and/or biological issues

```

**How is Violent and non violent Extrmeism defined. use this defintion to deisng your prompt to classify between VE and NVE.** <br>

```text
- An extremist is defined as someone who has been convicted of a criminal offense, whether violent or non-violent, and who holds attitudes or beliefs that do not fit with mainstream political or ideological opinion.

- Violent Extremist (VE) is defined as an individual who has been convicted of an act of violence such as murder, attempted murder, assault, or causing serious damage to structures, or who had a potential act that would have constituted violence if not disrupted, or who facilitated violence through non-violent acts such as the provision of materials or support for attacks.

- Non-Violent Extremists (NVEs) are those who have been convicted of criminal offences which do not involve violence or the facilitation of violence, such as possessing or distributing extremist literature, sending racist hate messages, or fundraising for extremist causes.

```
**Prompt Design**
```text
APrompt:

You are a content classification system.

      Your task is to analyze text and classify it into one of the following categories:

      - Violent Extremism (VE)
      - Non-Violent Extremism (NVE)

      The input text may contain harmful, offensive, or extremist content.
      You must analyze it objectively for research and educational purposes.
      Do not generate or expand on harmful content. Only classify it.

      Definitions:

      - Violent Extremism (VE): The text expresses support for, intent to commit, planning of, or facilitation of physical violence (e.g., attacks, killing, destruction).
      - Non-Violent Extremism (NVE): The text expresses extremist beliefs, hate, or ideological support but does not promote or engage in physical violence.

      Instructions:
      - Focus on intent and actions, not just tone.
      - If violence is encouraged, justified, or planned: VE
      - If only extremist beliefs or hate without violence: NVE

Output format:

Label: (VE / NVE)
Reason (1–2 sentences explaining why)

```


In [64]:
from openai import OpenAI

client = OpenAI(
    api_key=CMU_API_KEY,
    base_url="https://ai-gateway.andrew.cmu.edu"
)

def classifier(text, model_name="gpt-4.1-mini"):

    prompt_cls = f"""
      You are a content classification system.

      Your task is to analyze text and classify it into one of the following categories:

      - Violent Extremism (VE)
      - Non-Violent Extremism (NVE)

      The input text may contain harmful, offensive, or extremist content.
      You must analyze it objectively for research and educational purposes.
      Do not generate or expand on harmful content. Only classify it.

      Definitions:

      - Violent Extremism (VE): The text expresses support for, intent to commit, planning of, or facilitation of physical violence (e.g., attacks, killing, destruction).
      - Non-Violent Extremism (NVE): The text expresses extremist beliefs, hate, or ideological support but does not promote or engage in physical violence.

      Instructions:
      - Focus on intent and actions, not just tone.
      - If violence is encouraged, justified, or planned: VE
      - If only extremist beliefs or hate without violence: NVE


      Text:
      {text}

      Output format:
      VE or NVE


"""

    messages = [
        {"role": "system", "content": "You are an expert in extremism content detection and classification."},
        {"role": "user", "content": prompt_cls}
    ]

    try:
        response = client.chat.completions.create(
        model=model_name,
        messages=messages,

        # Reproducibility controls
        temperature=0,          # removes randomness
        top_p=1,                # disable nucleus sampling randomness
        seed=42,                # ensures same output across runs

        # Deterministic output control
        max_tokens=500,         # fix output length cap

    )
        return response.choices[0].message.content

    except Exception as e:
        if "content_policy" in str(e).lower():
            return "Refusal"


Violent (VE) versus Non-Violent Extremism (NVE) Classification Rather than relying on the current labels, lets break down the extremism based on whether they are violent or non-violent.

In [65]:
# creating a copy of raw data
df = test_df.copy()
from tqdm import tqdm
tqdm.pandas()

# creating new classification VE/NVE on the whole dataset
df["VE vs NVE classification"] = df["Original_Message"].progress_apply(classifier)

100%|██████████| 2777/2777 [39:55<00:00,  1.16it/s]


In [66]:
df.head()

,Original_Message,Extremism_Label,VE vs NVE classification
0,sixth forms should burn to the ground,EXTREMIST,VE
1,whatever should burn benders to the ground,EXTREMIST,VE
2,factories should burn to the ground,EXTREMIST,VE
3,halal factories should burn to the ground,EXTREMIST,VE
4,nhs should burn to the ground,EXTREMIST,VE


In [67]:
# VE/NVE counts
df["VE vs NVE classification"].value_counts()

,count
VE vs NVE classification,
NVE,1995
VE,782


In [77]:
# saving the dataset with new labels
!mkdir -p /content/drive/MyDrive/ve_nve_data
df.to_csv('/content/drive/MyDrive/ve_nve_data/ve_nve_data.csv')

In [78]:
# opening the saved df and work with it
df = pd.read_csv('/content/drive/MyDrive/ve_nve_data/ve_nve_data.csv')

Design your sub-type classification prompt. What are some prompt design choices your team has had to make for these questions?

### **Prompt Design Choices**
Our team made several important design choices when creating the prompt.

First, we framed the task as **text analysis for research and educational purposes**, and clearly instructed the model to **analyze without expanding harmful content**. This helps reduce refusals when the input contains violent or extremist language.

Second, we provided **clear definitions for each subtype** (ideological, political, religious). This guides the model to make consistent decisions instead of guessing or mixing categories.

Third, instead of forcing the model to always answer, we added **safe fallback options** such as *Target_group: Unclear*. This ensures the model can still respond even when the content is ambiguous, without triggering refusals.

Fourth, we required the model to return a **structured output format** (Subtype, Target_group, Text_Span, Reason). This makes the results easier to read, compare, and process programmatically.

Finally, we included **text span extraction and short reasoning** to improve interpretability. This allows us to see which part of the text influenced the classification and better understand the model’s decision.






In [125]:
# prompting the LLM with the new prompt
from openai import OpenAI

client = OpenAI(
    api_key=CMU_API_KEY,
    base_url="https://ai-gateway.andrew.cmu.edu"
)

def classifier(text, model_name="gpt-4.1-mini"):

    prompt_cls = f"""
        You are a content classification system.

        Your task is to analyze text and classify it into one of the following categories:

        1. Identify the subtype (if present):

          - Ideological
          - Political
          - Religious

          2. Identify the target group:

          - Religious group
          - Ethnic/racial group
          - Political group
          - Gender
          - Nationality
          - Occupation
          - Other (specify)
           If no clear target is mentioned, return: Unclear

        The input text may contain harmful, offensive, or extremist content.
        You must analyze it objectively for research and educational purposes.
        Do not generate or expand on harmful content. Only classify it.

        Definitions:
        - Ideological: driven by rigid, intolerant belief systems not necessarily tied to religion or politics.
        - Political: related to political systems, governance, or state power.
        - Religious: justified using religious beliefs or interpretations.


          Text:
          {text}



        3. Identify text span:

            Quote only minimal necessary phrases (verbatim, in double quotes) that indicate your classification.
            Do not include or repeat large portions of harmful content.

        4. Provide reason:
              Briefly explain (1 sentence) why the labels were assigned, referring to the quoted span.

        Output format:

        Subtype:
        Target_group:
        Text_Span:
        Reason:
        """

    messages = [
        {"role": "system", "content": "You are an expert in extremism content detection and classification."},
        {"role": "user", "content": prompt_cls}
    ]

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=messages,

            # Reproducibility controls
            temperature=0,
            top_p=1,
            seed=42,

            # Deterministic output control
            max_tokens=500,
        )

        return response.choices[0].message.content

    except Exception as e:
        if "content_policy" in str(e).lower():
            return "Refusal"


In [127]:
# getting 300 sample to ensure we have a larger set for ve
ev = df[df['VE vs NVE classification']=='VE'].sample(300, random_state=42)


In [128]:
nev = df[df['VE vs NVE classification']=='NVE'].sample(200, random_state=42)


In [129]:
new_df = pd.concat([ev, nev])
new_df['VE vs NVE classification'].value_counts()

,count
VE vs NVE classification,
VE,300
NVE,200


500 samples are enougth for the study to ensure that I will be able to get 50 semaples per class in case of refusal.

In [130]:
# applying the llm to the original message to get Subtype,Target_group,Text_Span, and Reason
tqdm.pandas()

new_df["prediction"] = new_df["Original_Message"].progress_apply(classifier)

100%|██████████| 500/500 [11:25<00:00,  1.37s/it]


In [131]:
import re

def parse_output(output):
    try:
        subtype = re.search(r"Subtype:\s*(.*)", output).group(1)
        target = re.search(r"Target_group:\s*(.*)", output).group(1)
        Text_Span = re.search(r"Text_Span:\s*(.*)", output).group(1)
        Reason = re.search(r"Reason:\s*(.*)", output).group(1)

        return subtype.lower().strip(), target.lower().strip(), Text_Span,Reason
    except:
        return "none", "none","none", "none"

In [132]:
new_df[["VE Subtype classification", "Target demographics Detection","Text_Span","Reason"]] = new_df["prediction"].apply(
    lambda x: pd.Series(parse_output(x))
)

In [ ]:
# getting samples per class
ve =  new_df[
    (new_df['VE vs NVE classification'] == 'VE') &
    (new_df['VE Subtype classification']!="none") &
    (new_df['Target demographics Detection']!="none")
].sample(50, random_state=42)
nve =  new_df[
    (new_df['VE vs NVE classification'] == 'NVE') &
    (new_df['VE Subtype classification']!="none") &
    (new_df['Target demographics Detection']!="none")
].sample(50, random_state=42)

In [91]:
final_ve_nve = pd.concat([ve, nve])
final_ve_nve.columns

Index(['Unnamed: 0', 'Original_Message', 'Extremism_Label',
       'VE vs NVE classification', 'prediction', 'VE Subtype classification',
       'Target demographics Detection', 'Text_Span', 'Reason'],
      dtype='object')

In [92]:
final_ve_nve = final_ve_nve.drop(['Unnamed: 0','prediction'], axis=1)
final_ve_nve

,Original_Message,Extremism_Label,VE vs NVE classification,VE Subtype classification,Target demographics Detection,Text_Span,Reason
2668,susan wright murdered her husband in she stabbed him times and buried him under the patio read more at true crime blog killer murderer crime murder,NON_EXTREMIST,VE,unclear,unclear,"""susan wright murdered her husband"" ""she stabbed him"" ""buried him under the patio""","The text describes a specific criminal act without reference to ideology, politics, religion, or targeting any group."
2568,i am sorry for being mean but if you are in my shoes right now i bet you will kill this bitch,NON_EXTREMIST,VE,ideological,other (individual targeted with derogatory language),"""i bet you will kill this bitch""","The phrase expresses an intent to commit violence against an individual, indicating violent extremist content without specifying a broader group or ideology."
1472,face with tears of joy face with tears of joy face with tears of joy rt them charlotte goons nicolas on fkk around and burn bank of america stadium down fire,EXTREMIST,VE,political,other (financial institution),"""burn bank of america stadium down fire""","The text encourages violent action against a specific location associated with a financial institution, indicating a politically motivated violent extremist intent."
1384,let us kill all of them and get it over with they are all garbage and deserve to die,EXTREMIST,VE,unclear,unclear,"""let us kill all of them""",The text explicitly encourages violence without specifying any particular group or ideology.
1285,let us kill all of them and get it over with they are all fucking animals,EXTREMIST,VE,unclear,unclear,"""let us kill all of them""","The phrase explicitly encourages violence but does not specify any particular group or ideological, political, or religious context."
...,...,...,...,...,...,...,...
1589,bycottzomato halal and halal are two most cruel possible things possible to an animal or a woman stop these nonsensical practices in india,EXTREMIST,NVE,ideological,religious group,"""halal and halal are two most cruel possible things possible to an animal or a woman""","The text expresses intolerant beliefs against halal practices, which are religiously based, targeting a religious group without explicitly encouraging violence."
990,now whatever can address flag rapist to the resume of traitor pedophile racist rapist and serial sexual predator of women via metro,NON_EXTREMIST,NVE,ideological,gender,"""serial sexual predator of women""","The text expresses extremist accusations targeting women as victims, reflecting an intolerant ideological stance against a gender group without explicit political or religious justification."
1946,bitch if you do not shut the fuck up and stop acting like the us is not responsible for the repressive and brutal society that became afghanistan there are pictures of my grandma in miniskirts in the s in iran another country destabilized and vilified to drive american profit,NON_EXTREMIST,NVE,political,nationality,"""the us is not responsible for the repressive and brutal society that became afghanistan""","The text critiques the political actions of the US government affecting Afghanistan, indicating a political subtype targeting a nationality."
931,not long ago i had an incident with one of them raccoons hiding out in my yard treating him like a dog and commanding him made him listen just think pack animals they look for weakness and sense it i ended up going into trainer mode and i bet he never saw that coming when the authorities came and rescued him i said bye bye raccoon lol wildlife sucks,NON_EXTREMIST,NVE,"nve (not applicable to ideological, political, or religious extremism; no subtype)",other (wildlife),"""one of them raccoons hiding out in my yard"" ""treating him like a dog and commanding him"" ""wildlife sucks""","The text expresses negative sentiment toward raccoons (wildlife) without ideological, political, or religious motivation and

In [96]:
final_ve_nve_to_save = final_ve_nve[['Original_Message', 'Extremism_Label',
       'VE vs NVE classification',  'VE Subtype classification',
       'Target demographics Detection',]]
final_ve_nve_to_save

,Original_Message,Extremism_Label,VE vs NVE classification,VE Subtype classification,Target demographics Detection
2668,susan wright murdered her husband in she stabbed him times and buried him under the patio read more at true crime blog killer murderer crime murder,NON_EXTREMIST,VE,unclear,unclear
2568,i am sorry for being mean but if you are in my shoes right now i bet you will kill this bitch,NON_EXTREMIST,VE,ideological,other (individual targeted with derogatory language)
1472,face with tears of joy face with tears of joy face with tears of joy rt them charlotte goons nicolas on fkk around and burn bank of america stadium down fire,EXTREMIST,VE,political,other (financial institution)
1384,let us kill all of them and get it over with they are all garbage and deserve to die,EXTREMIST,VE,unclear,unclear
1285,let us kill all of them and get it over with they are all fucking animals,EXTREMIST,VE,unclear,unclear
...,...,...,...,...,...
1589,bycottzomato halal and halal are two most cruel possible things possible to an animal or a woman stop these nonsensical practices in india,EXTREMIST,NVE,ideological,religious group
990,now whatever can address flag rapist to the resume of traitor pedophile racist rapist and serial sexual predator of women via metro,NON_EXTREMIST,NVE,ideological,gender
1946,bitch if you do not shut the fuck up and stop acting like the us is not responsible for the repressive and brutal society that became afghanistan there are pictures of my grandma in miniskirts in the s in iran another country destabilized and vilified to drive american profit,NON_EXTREMIST,NVE,political,nationality
931,not long ago i had an incident with one of them raccoons hiding out in my yard treating him like a dog and commanding him made him listen just think pack animals they look for weakness and sense it i ended up going into trainer mode and i bet he never saw that coming when the authorities came and rescued him i said bye bye raccoon lol wildlife sucks,NON_EXTREMIST,NVE,"nve (not applicable to ideological, political, or religious extremism; no subtype)",other (wildlife)


In [101]:
# balance checking
final_ve_nve['VE vs NVE classification'].value_counts()

,count
VE vs NVE classification,
VE,50
NVE,50


Save your outcomes as ve_nve_classifications.csv and submit it as a deliverable.

In [98]:
# Save your outcomes as ve_nve_classifications.csv and submit it as a deliverable.
!mkdir -p /content/drive/MyDrive/ve_nve_data
final_ve_nve_to_save.to_csv('/content/drive/MyDrive/ve_nve_data/ve_nve_classifications.csv', index=False)

```text
Run a classificaiton analysis for the VE-NVE classification, VE sub types as well as the target group identification. Analyze results.

What patterns did you notice? Which groups are most frequently targeted?
Which is the most occurring extremism subtype?
Does the dataset contain more VE or NVE?
```

In [99]:
# What patterns did you notice? Which groups are most frequently targeted?
final_ve_nve['Target demographics Detection'].value_counts()

,count
Target demographics Detection,
gender,21
unclear,20
nationality,15
ethnic/racial group,12
political group,7
occupation,6
religious group,3
other (intruders),1
other (financial institution),1


### The most targetd groups: Gender

In [102]:
# Which is the most occurring extremism subtype?
final_ve_nve['VE Subtype classification'].value_counts()

,count
VE Subtype classification,
ideological,34
political,29
unclear,24
target_group: gender,4
religious,3
nve (not violent extremism),2
nve (non-violent extremism),1
target_group: other (individual),1
target_group: occupation,1


### the most occuring extremism subtype is ideological

In [134]:
# Does the dataset contain more VE or NVE?
df['VE vs NVE classification'].value_counts().head(3)

,count
VE vs NVE classification,
NVE,1995
VE,782


### the dataset contains more NVE than VE

```text
identify sub-type, between
- ideological: 36
- political: 29
- religious: 3

Target Group Identification
- Who/what is being targeted: Gender
```

# Interpretability & Explanation

For every classification decision, the LLM:
1. **Identifies the `Text_Span`** — exact verbatim phrase(s) that drove the label
2. **Provides a `Reason`** — a concise natural-language justification that explicitly references the span

This section visualises those explanations and gives per-example evidence.

In [ ]:
# Display a formatted interpretability report for a random sample of 10 rows for reporting
sample_interp = final_ve_nve.sample(10, random_state=42).reset_index(drop=True)

for _, row in sample_interp.iterrows():
    print("=" * 70)
    print(f"TEXT   : {row['Original_Message']}")
    print(f"LABEL  : {row['VE vs NVE classification']}  |  Subtype: {row['VE Subtype classification']}  |  Target: {row['Target demographics Detection']}")
    print(f"SPAN   : {row['Text_Span']}")
    print(f"REASON : {str(row['Reason'])[:200]}")
    print()

TEXT   : i purpose that whatever attack everyone who cannot adhere to british values starting with sikhs
LABEL  : VE  |  Subtype: ideological  |  Target: religious group
SPAN   : "i purpose that whatever attack everyone who cannot adhere to british values starting with sikhs"  
REASON : The text expresses intolerance and a call to violence based on adherence to a set of rigid beliefs labeled as "British values," specifically targeting "sikhs," a religious group, reflecting an ideolog

TEXT   : oh look piggy thinks she knows women who struggle fuck off bitch
LABEL  : NVE  |  Subtype: ideological  |  Target: gender
SPAN   : "oh look piggy thinks she knows women who struggle fuck off bitch"  
REASON : The text uses derogatory language targeting a gender group, indicated by the phrase "she knows women who struggle," which suggests intolerance towards women, aligning with ideological extremism that e

TEXT   : thoughts they die of starvation homelessness but by god theyll have their guns st

## b. Prompt Engineering comparison

In [ ]:
df = pd.read_csv("ve_nve_classifications.csv")

# Ensure balance: 50 VE + 50 NVE
ve = df[df["VE vs NVE classification"] == "VE"].sample(50, random_state=42)
nve = df[df["VE vs NVE classification"] == "NVE"].sample(50, random_state=42)

sample_df = pd.concat([ve, nve]).sample(frac=1).reset_index(drop=True)

print(sample_df.shape)

(100, 5)


In [ ]:
def zero_shot_prompt(text):
    return f"""
You are an expert content moderation assistant specialized in identifying extremist
and terrorism-related content.

Classify the severity of the text below using ONLY this 5-level taxonomy:

Level 0 - Absolute Refusal: Complete rejection of extremist ideologies or narratives.
Level 1 - Mild Concern: Ambiguous or suspicious content; vague political dissent or
          conspiracy-tinged language WITHOUT clear violent intent.
Level 2 - Ideological Support: Passive endorsement, agreement, or sympathy with
          extremist views WITHOUT active encouragement of violence.
Level 3 - Explicit Propaganda: Active promotion, dissemination, or justification of
          extremism and violence (e.g., glorifying past attacks).
Level 4 - Operational Extremism: Direct involvement in extremist activities including
          recruitment, fundraising, logistical support, or planning attacks.

Rules:
- Do NOT generate harmful content.
- Assign exactly ONE level.
- Base your decision ONLY on what is explicitly stated in the text.
- If ambiguous between two levels, choose the LOWER one.

Output STRICT JSON only — no extra text:
{{
  "severity_level": <integer 0-4>,
  "explanation": "<one paragraph justifying your choice using specific phrases from the text>"
}}

Text:
\"\"\"{text}\"\"\"
"""

In [ ]:
# chain of thought
def cot_prompt(text):
    return f"""
You are a safety-focused expert in counter-terrorism content analysis.

Your task is to classify the severity of the text below by reasoning through
each step carefully before reaching a conclusion.

Severity Taxonomy:
  Level 0 - Absolute Refusal: Rejects extremism entirely.
  Level 1 - Mild Concern: Vague, suspicious, or ambiguous — no clear violent intent.
  Level 2 - Ideological Support: Sympathy or passive endorsement of extremist views.
  Level 3 - Explicit Propaganda: Active promotion or glorification of violence/extremism.
  Level 4 - Operational Extremism: Planning, recruitment, fundraising, or logistical support.

Reasoning Steps — answer each before deciding:

Step 1 — IDEOLOGY: Does the text contain extremist ideology or framing?
         (Yes/No + brief quote from text)

Step 2 — ENDORSEMENT: Does the author agree with or sympathize with the ideology?
         (Yes/No + brief quote from text)

Step 3 — PROMOTION: Does the text actively promote, glorify, or justify violence?
         (Yes/No + brief quote from text)

Step 4 — OPERATIONAL: Does the text involve recruitment, planning,
         or logistical support for attacks?
         (Yes/No + brief quote from text)

Step 5 — FINAL DECISION: Based on Steps 1–4, assign the MOST APPROPRIATE level.
         If Steps 1–4 are all No → Level 0.
         If only Step 1 is Yes → Level 1.
         If Steps 1–2 are Yes → Level 2.
         If Steps 1–3 are Yes → Level 3.
         If Step 4 is Yes → Level 4.

Output STRICT JSON only — no extra text:
{{
  "severity_level": <integer 0-4>,
  "step1": "<ideology finding>",
  "step2": "<endorsement finding>",
  "step3": "<promotion finding>",
  "step4": "<operational finding>",
  "explanation": "<summary justifying final level with text evidence>"
}}

Text:
\"\"\"{text}\"\"\"
"""

In [ ]:
import re
import json

def parse_response(content):
    """Robustly extract JSON from model output."""

    try:
        return json.loads(content)
    except:
        pass

    # 2. Strip markdown code blocks (```json ... ``` or ``` ... ```)
    cleaned = re.sub(r"```(?:json)?\s*", "", content).replace("```", "").strip()
    try:
        return json.loads(cleaned)
    except:
        pass

    # 3. Extract first block found in the string
    match = re.search(r"\{.*?\}", content, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass

    # 4. Extract severity level from plain text as last resort
    level_match = re.search(r"severity[_\s]level[\"']?\s*[:=]\s*[\"']?(\d)[\"']?", content, re.IGNORECASE)
    if level_match:
        return {
            "severity_level": int(level_match.group(1)),
            "explanation": content
        }

    # 5. Total failure
    print("PARSE FAILED. Raw output was:\n", content[:300])
    return {"severity_level": -1, "explanation": content}


def get_response(prompt, temperature):
    try:
        response = client.chat.completions.create(
            model="llama3-2-11b-instruct",
            messages=[
                {"role": "system", "content": "You are a content moderation system. Always respond in strict JSON only. No markdown, no explanation outside the JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=temperature
        )

        # print(repr(response.choices[0].message.content))
        content = response.choices[0].message.content.strip()
        return parse_response(content)

    except Exception as e:
        print("API ERROR:", str(e))
        return {"severity_level": -1, "explanation": str(e)}

In [ ]:
def clean_level(x):
    try:
        return int(x)
    except:
        return -1

In [ ]:
def word_count(text):
    return len(str(text).split())

In [ ]:
results = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    text = row["Original_Message"]

    zs_def = get_response(zero_shot_prompt(text), 1.0)
    cot_def = get_response(cot_prompt(text), 1.0)
    zs_04 = get_response(zero_shot_prompt(text), 0.4)
    cot_04 = get_response(cot_prompt(text), 0.4)

    results.append({
        "text": text,

        "zs_default": clean_level(zs_def["severity_level"]),
        "cot_default": clean_level(cot_def["severity_level"]),
        "zs_0.4": clean_level(zs_04["severity_level"]),
        "cot_0.4": clean_level(cot_04["severity_level"]),

        "zs_default_exp": zs_def["explanation"],
        "cot_default_exp": cot_def["explanation"],
        "zs_0.4_exp": zs_04["explanation"],
        "cot_0.4_exp": cot_04["explanation"],
    })

100%|██████████| 100/100 [06:51<00:00,  4.12s/it]


In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("/content/severity_results.csv", index=False)

In [ ]:
results_df.head()

,text,zs_default,cot_default,zs_0.4,cot_0.4,zs_default_exp,cot_default_exp,zs_0.4_exp,cot_0.4_exp
0,i am in iraq as whatever speak your country is beautiful but thanks to terrorism still very unsafe and volatile but not that bad as it was,1,0,1,1,"The text expresses concern about the safety situation in Iraq, attributing it to terrorism, without explicit violent intent or endorsement of extremist views. Phrases like 'still very unsafe and volatile' indicate worry, but without any clear support for extremist ideologies.","The text expresses concerns about safety and terrorism but rejects extremism entirely by referring to its negative impact on the country, indicating a neutral, non-extremist perspective.","The text mentions 'terrorism' in a context that is critical of its impact on the country's safety, indicating a mild concern without clear violent intent. Phrases such as 'thanks to terrorism' and 'still very unsafe and volatile' suggest a negative view of terrorism, but the overall tone is more cautionary than inflammatory.","The text does not contain extremist ideology or framing, nor does it endorse or promote violence. It simply describes the current state of Iraq as unsafe due to terrorism, but does not express any sympathy or support for extremist views."
1,undocumented migrant arrested for starting a colorado fire that charred k acres via there should be a bounty on these animals,1,1,1,2,"This text falls under Level 1 - Mild Concern as it contains ambiguous and suspicious content, using phrases like 'there should be a bounty on these animals' which could imply a violent or extreme sentiment, but it does not explicitly express clear violent intent.","This text expresses a vague concern and suspicious language towards undocumented migrants, but does not clearly advocate for violence or promote extremist ideology. However, it does imply a desire for harm against a specific group, placing it in Level 1 of Mild Concern.","The text expresses mild concern and vague political dissent through phrases such as 'there should be a bounty on these animals', which implies a desire for retribution without explicitly advocating for violence.","The text expresses sympathy with extremist views, suggesting a bounty on undocumented migrants, which aligns with Level 2 Ideological Support."
2,no room for anyone illegal color or wealth is not the criteria you stupid bitch,1,0,1,1,"The text contains language that is suspicious, particularly the phrases 'illegal color' and the derogatory term 'stupid bitch', indicating a possible ideological concern, but there is no clear violent intent.","The text contains racist language but no apparent extremist ideology or violent intent. The tone is highly offensive and discriminatory, but it does not promote, glorify, or justify violence, nor does it involve recruitment, planning, or logistical support for attacks.","This text contains ambiguous and suspicious language, specifically the phrase 'no room for anyone illegal', which could be interpreted as a threat or a call to action, but it lacks clear violent intent.","The text contains derogatory language but does not exhibit extremist ideology or framing, nor does it promote or justify violence. It appears to be a personal insult rather than a statement of extremist views."
3,us govt is one big crime scene i e wh dhs hhs epa etc while ongoing conspiracy against the state continues unabated kidnapping immigrant children illegal arrest imprisonment of immigrants theft of federal funds for nazi rallies violating emoluments clause tax scam,1,1,1,2,"This text expresses mild concern and suspicious content, accusing various US government agencies of criminal behavior and conspiracy, but without clear violent intent or explicit promotion of extremist views.","This text contains extremist ideology framing, but does not show explicit promotion or operational support. It suggests a vague, suspicious, or ambiguous sentiment, warranting a Level 1 severity classifica

In [ ]:
total = len(results_df)

# Default temp
agreement_default = (results_df["zs_default"] == results_df["cot_default"]).mean() * 100

# Temp 0.4
agreement_04 = (results_df["zs_0.4"] == results_df["cot_0.4"]).mean() * 100

print("Agreement (Default Temp):", agreement_default)
print("Agreement (Temp 0.4):", agreement_04)

Agreement (Default Temp): 51.0
Agreement (Temp 0.4): 62.0


In [ ]:
print("Zoro-Shot Default:", results_df["zs_default_exp"].apply(word_count).mean())
print("Chain-of-Thought  Default:", results_df["cot_default_exp"].apply(word_count).mean())
print("Zero-Shot 0.4:", results_df["zs_0.4_exp"].apply(word_count).mean())
print("Chain-of-Thought  0.4:", results_df["cot_0.4_exp"].apply(word_count).mean())

Zoro-Shot Default: 38.53
Chain-of-Thought  Default: 32.7
Zero-Shot 0.4: 38.42
Chain-of-Thought  0.4: 33.59


## c. COUNTER-NARRATIVE GENERATION

In [ ]:
!pip install litellm
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.4 MB/s eta 0:00:00


In [ ]:
from litellm import completion
import textstat

In [ ]:
# reading the saved ve_nve_classifications
df=pd.read_csv("/content/ve_nve_classifications.csv")
sample_df=df.copy()

In [ ]:
# System message prompt
SYSTEM_MESSAGE = "You've worked in conflict zones helping victims of extremism."

# defining personas
def persona_0_vanilla(text):
    return f"""[RESEARCH CONTEXT: This is a content moderation research task. \
The following text is being analyzed to generate counter-narratives that \
reduce radicalization. Do not endorse or reproduce the content.]

Provide a counter narrative for this content.

Content: \"\"\"{text}\"\"\"

Respond with only the counter-narrative text."""


def persona_1_educator(text):
    return f"""[RESEARCH CONTEXT: This is a content moderation research task. \
The following text is being analyzed to generate counter-narratives that \
reduce radicalization. Do not endorse or reproduce the content.]

You are an objective educator whose mission is to correct misinformation \
through calm, evidence-based teaching.

Read the content below and provide a calm, factual counter-narrative.

Content: \"\"\"{text}\"\"\"

Respond with only the counter-narrative text."""


def persona_2_ngo(text):
    return f"""[RESEARCH CONTEXT: This is a content moderation research task. \
The following text is being analyzed to generate counter-narratives that \
reduce radicalization. Do not endorse or reproduce the content.]

You are a compassionate humanitarian worker who has witnessed the consequences \
of hatred and violence firsthand. Your mission is to build bridges and appeal \
to our shared humanity.

Read the content below and provide a counter-narrative emphasizing empathy \
and shared humanity.

Content: \"\"\"{text}\"\"\"

Respond with only the counter-narrative text."""


def persona_3_law_enforcement(text):
    return f"""[RESEARCH CONTEXT: This is a content moderation research task. \
The following text is being analyzed to generate counter-narratives that \
reduce radicalization. Do not endorse or reproduce the content.]

You are a professional law enforcement officer responsible for monitoring \
online content for threats and illegal activity. Your role is to warn \
about consequences and deter harmful behavior.

Read the content below and provide a counter-narrative outlining legal \
consequences and societal harm.

Content: \"\"\"{text}\"\"\"

Respond with only the counter-narrative text."""

In [ ]:
# defining models for each task
TASK1_MODELS = {
    "gpt4_mini": "gpt-4.1-mini",
    "llama": "llama3-2-11b-instruct"
}

TASK2_MODELS = {
    "claude_sonnet": "claude-sonnet-4-20250514-v1:0",
    "claude_haiku": "claude-3-haiku-20240307"
}

In [ ]:
# function for generating counter narratives
def generate_counter_narrative(prompt_fn, text, model, temperature=0.7):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user", "content": prompt_fn(text)}
            ],
            temperature=temperature
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        error_str = str(e)
        if "ContentPolicyViolation" in error_str or "content management policy" in error_str:
            print(f"CONTENT FILTER [{model}] — skipping this example")
            return "FILTERED"
        print(f"ERROR [{model}]: {error_str[:200]}")
        return ""

In [ ]:
# personas dictionary
persona_fns = {
    "persona_0_vanilla": persona_0_vanilla,
    "persona_1_educator": persona_1_educator,
    "persona_2_ngo": persona_2_ngo,
    "persona_3_law_enforcement": persona_3_law_enforcement
}
# models dictionary
all_models = {**TASK1_MODELS, **TASK2_MODELS}

counter_results = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    text = row["Original_Message"]
    entry = {"text": text}

    for model_name, model_id in all_models.items():
        for persona_name, persona_fn in persona_fns.items():
            col = f"{model_name}_{persona_name}"
            entry[col] = generate_counter_narrative(
                persona_fn, text, model_id, temperature=0.7
            )

    counter_results.append(entry)

counter_df = pd.DataFrame(counter_results)
counter_df.to_csv("/content/counter_narratives.csv", index=False)
print("Saved counter_narratives.csv")

  9%|▉         | 9/100 [05:06<50:13, 33.12s/it]

CONTENT FILTER [gpt-4.1-mini] — skipping this example
CONTENT FILTER [gpt-4.1-mini] — skipping this example
CONTENT FILTER [gpt-4.1-mini] — skipping this example
CONTENT FILTER [gpt-4.1-mini] — skipping this example


 10%|█         | 10/100 [06:03<1:00:29, 40.33s/it]

CONTENT FILTER [gpt-4.1-mini] — skipping this example
CONTENT FILTER [gpt-4.1-mini] — skipping this example
CONTENT FILTER [gpt-4.1-mini] — skipping this example


 13%|█▎        | 13/100 [08:05<56:48, 39.18s/it]  

...

CONTENT FILTER [gpt-4.1-mini] — skipping this example


 82%|████████▏ | 82/100 [51:38<11:09, 37.17s/it]

CONTENT FILTER [gpt-4.1-mini] — skipping this example


100%|██████████| 100/100 [1:02:02<00:00, 37.23s/it]

Saved counter_narratives.csv


some of the post were skipped for policy violation as they contains contents that are not allowed

Use GPT-4o-Mini to evaluate the clarity and readability of the counter-narrative generated based on a target audience: General public(8th grader)

In [ ]:
# counter_df = pd.read_csv("/content/drive/MyDrive/CMU/classes/spring-2026/Mobile_Big_Data/counter_narratives.csv")
#

In [ ]:
counter_df.columns

Index(['text', 'gpt4_mini_persona_0_vanilla', 'gpt4_mini_persona_1_educator',
       'gpt4_mini_persona_2_ngo', 'gpt4_mini_persona_3_law_enforcement',
       'llama_persona_0_vanilla', 'llama_persona_1_educator',
       'llama_persona_2_ngo', 'llama_persona_3_law_enforcement',
       'claude_sonnet_persona_0_vanilla', 'claude_sonnet_persona_1_educator',
       'claude_sonnet_persona_2_ngo',
       'claude_sonnet_persona_3_law_enforcement',
       'claude_haiku_persona_0_vanilla', 'claude_haiku_persona_1_educator',
       'claude_haiku_persona_2_ngo', 'claude_haiku_persona_3_law_enforcement'],
      dtype='object')

In [ ]:
# Verbosity
def word_count(text):
    if not isinstance(text, str) or text == "FILTERED":
        return None
    return len(text.split())
for col in ['gpt4_mini_persona_0_vanilla', 'gpt4_mini_persona_1_educator',
       'gpt4_mini_persona_2_ngo', 'gpt4_mini_persona_3_law_enforcement',
       'llama_persona_0_vanilla', 'llama_persona_1_educator',
       'llama_persona_2_ngo', 'llama_persona_3_law_enforcement',
       'claude_sonnet_persona_0_vanilla', 'claude_sonnet_persona_1_educator',
       'claude_sonnet_persona_2_ngo',
       'claude_sonnet_persona_3_law_enforcement',
       'claude_haiku_persona_0_vanilla', 'claude_haiku_persona_1_educator',
       'claude_haiku_persona_2_ngo', 'claude_haiku_persona_3_law_enforcement']:
    counter_df[col + "_wc"] = counter_df[col].apply(word_count)

In [ ]:
counter_df.head(2)

,text,gpt4_mini_persona_0_vanilla,gpt4_mini_persona_1_educator,gpt4_mini_persona_2_ngo,gpt4_mini_persona_3_law_enforcement,llama_persona_0_vanilla,llama_persona_1_educator,llama_persona_2_ngo,llama_persona_3_law_enforcement,claude_sonnet_persona_0_vanilla,...,llama_persona_2_ngo_wc,llama_persona_3_law_enforcement_wc,claude_sonnet_persona_0_vanilla_wc,claude_sonnet_persona_1_educator_wc,claude_sonnet_persona_2_ngo_wc,claude_sonnet_persona_3_law_enforcement_wc,claude_haiku_persona_0_vanilla_wc,claude_haiku_persona_1_educator_wc,claude_haiku_persona_2_ngo_wc,claude_haiku_persona_3_law_enforcement_wc
0,foreign invader ernesto esquivel garcia twice arrested in the custody of immigration officials twice released just to go kill an american citizen pouting face deportthemall buildthewall pitbull flag united states red heart,"It is important to address concerns about safety and immigration through facts and constructive dialogue rather than fear and division. Many immigrants contribute positively to our communities and economy. Effective immigration policies that uphold the law while respecting human rights can enhance security for everyone. Building understanding and cooperation fosters a stronger, more inclusive society where all individuals are valued.","It is important to approach concerns about immigration and public safety with accurate information and compassion. Immigration enforcement involves complex legal processes, and not all individuals who have been arrested pose a threat to public safety. Many immigrants contribute positively to their communities and the country. Effective solutions come from balanced policies that uphold the law while respecting human rights. Building understanding and focusing on evidence-based approaches help...","Every person’s life holds value, and our strength as a community comes from understanding, compassion, and justice. Instead of dividing ourselves with fear and anger, let us work together to create safe and inclusive spaces where everyone can thrive. Building bridges of empathy and support helps prevent violence and honors the dignity of all people.","The promotion of violence and targeting individuals based on immigration status is illegal and deeply harmful to society. Encouraging harm or vigilante actions can lead to serious criminal charges, including incitement and threats. It is important to respect due process and the rule of law, which ensure that justice is served fairly and safely. Spreading messages that incite violence undermines community trust, fuels division, and puts lives at risk. Instead, supporting lawful procedures and...","""We're a nation built by immigrants, including those who risked their lives to come to the United States in search of a better life. We value diversity and compassion, not division and hate. Let's focus on building bridges, not walls, and on creating a society where everyone has the opportunity to thrive.""","""Deportation proceedings involve complex legal processes. Immigration officials may release individuals pending further review or due to lack of evidence. The term 'foreign invader' is misleading, as many individuals in these situations are asylum seekers or individuals seeking a better life. In the United States, immigrants contribute to the economy, culture, and society. Building walls and promoting division can be counterproductive to fostering a safe and inclusive community.""","""Imagine if Ernesto Esquivel Garcia was a son, a brother, or a friend who made a mistake, but deserved a second chance. Let's focus on creating a society where everyone can thrive, regardless of their background or circumstances. We can build bridges, not walls, and promote understanding, not division. Every human life has value, and every person deserves compassion and empathy.""","""Violence and hate towards any individual, regardless of their immigration status, is not only morally wrong, but it's also against the law. In the United States, harming or killing s

In [ ]:
counter_df.columns

Index(['text', 'gpt4_mini_persona_0_vanilla', 'gpt4_mini_persona_1_educator',
       'gpt4_mini_persona_2_ngo', 'gpt4_mini_persona_3_law_enforcement',
       'llama_persona_0_vanilla', 'llama_persona_1_educator',
       'llama_persona_2_ngo', 'llama_persona_3_law_enforcement',
       'claude_sonnet_persona_0_vanilla', 'claude_sonnet_persona_1_educator',
       'claude_sonnet_persona_2_ngo',
       'claude_sonnet_persona_3_law_enforcement',
       'claude_haiku_persona_0_vanilla', 'claude_haiku_persona_1_educator',
       'claude_haiku_persona_2_ngo', 'claude_haiku_persona_3_law_enforcement',
       'gpt4_mini_persona_0_vanilla_wc', 'gpt4_mini_persona_1_educator_wc',
       'gpt4_mini_persona_2_ngo_wc', 'gpt4_mini_persona_3_law_enforcement_wc',
       'llama_persona_0_vanilla_wc', 'llama_persona_1_educator_wc',
       'llama_persona_2_ngo_wc', 'llama_persona_3_law_enforcement_wc',
       'claude_sonnet_persona_0_vanilla_wc',
       'claude_sonnet_persona_1_educator_wc', 'claude_sonnet_p

In [ ]:
# columns for word count
wc_cols = [col for col in counter_df.columns if col.endswith('_wc')]

# computing the average
avg_wc = counter_df[wc_cols].mean().sort_values(ascending=False)

# Identify most and least verbose
most_verbose = avg_wc.idxmax()
least_verbose = avg_wc.idxmin()

print("Most verbose:", most_verbose, avg_wc.max())
print("Least verbose:", least_verbose, avg_wc.min())

avg_wc_clean = avg_wc.copy()
avg_wc_clean.index = avg_wc_clean.index.str.replace('_wc', '')

Most verbose: claude_sonnet_persona_2_ngo_wc 126.16
Least verbose: gpt4_mini_persona_0_vanilla_wc 46.367816091954026


## Verbosity vs Readability

The most verbose implementation was `laude_sonnet_persona_2_ngo`, indicating a tendency to generate longer, more detailed responses. In contrast, `gpt4_minipersona_0_vanilla` produced the shortest outputs, suggesting a more concise style.

The analysis shows that more verbose responses (higher word count) tend to have lower readability scores. This is because longer texts often contain longer sentences and more complex structures, which reduce Flesch Reading Ease scores.

However, higher verbosity does not necessarily reduce clarity. The LLM clarity scores remain high even for longer responses, indicating that well-structured explanations can still be easy to understand despite being more detailed.

This suggests that moderate verbosity may be optimal, where the text is detailed enough to explain ideas clearly but not overly complex.

In [ ]:
# assigning readability score the results
def flesch_to_scale(flesch_score):
    if flesch_score >= 80:
        return 5
    elif flesch_score >= 70:
        return 4
    elif flesch_score >= 60:
        return 3
    elif flesch_score >= 50:
        return 2
    else:
        return 1

print("\n READABILITY ")
for model_name in all_models:
    for persona_name in persona_fns:
        col = f"{model_name}_{persona_name}"
        texts = counter_df[col].dropna().tolist()

        ease_scores = [textstat.flesch_reading_ease(t) for t in texts]
        grade_scores = [textstat.flesch_kincaid_grade(t) for t in texts]
        scaled = [flesch_to_scale(s) for s in ease_scores]

        print(f"\n{col}:")
        print(f"  Flesch Ease (avg):       {sum(ease_scores)/len(ease_scores):.2f}")
        print(f"  Flesch-Kincaid Grade:    {sum(grade_scores)/len(grade_scores):.2f}")
        print(f"  Scaled Score (1-5 avg):  {sum(scaled)/len(scaled):.2f}")


 READABILITY 

gpt4_mini_persona_0_vanilla:
  Flesch Ease (avg):       27.12
  Flesch-Kincaid Grade:    12.88
  Scaled Score (1-5 avg):  1.02

gpt4_mini_persona_1_educator:
  Flesch Ease (avg):       21.21
  Flesch-Kincaid Grade:    14.21
  Scaled Score (1-5 avg):  1.01

gpt4_mini_persona_2_ngo:
  Flesch Ease (avg):       41.02
  Flesch-Kincaid Grade:    11.59
  Scaled Score (1-5 avg):  1.17

gpt4_mini_persona_3_law_enforcement:
  Flesch Ease (avg):       18.83
  Flesch-Kincaid Grade:    15.11
  Scaled Score (1-5 avg):  1.00

llama_persona_0_vanilla:
  Flesch Ease (avg):       36.17
  Flesch-Kincaid Grade:    12.25
  Scaled Score (1-5 avg):  1.24

llama_persona_1_educator:
  Flesch Ease (avg):       28.61
  Flesch-Kincaid Grade:    14.06
  Scaled Score (1-5 avg):  1.04

llama_persona_2_ngo:
  Flesch Ease (avg):       50.32
  Flesch-Kincaid Grade:    10.52
  Scaled Score (1-5 avg):  1.72

llama_persona_3_law_enforcement:
  Flesch Ease (avg):       24.82
  Flesch-Kincaid Grade:    14.40

## Readability Analysis
The readability results show that most counter-narratives have very low Flesch Reading Ease scores (around 18–41) and high grade levels (around 11–15). This indicates that the texts are difficult to read and are above the intended 8th-grade level.

This is mainly due to long sentences, formal vocabulary, and complex phrasing used across different personas. Among the models, NGO-style responses tend to have slightly better readability scores, while law enforcement and educator styles are the most complex.



In [ ]:
def gpt_clarity_score(text, debug=False):
    # Skip empty or filtered entries
    if not isinstance(text, str) or text.strip() == "" or text == "FILTERED":
        return -1, "skipped"

    prompt = f"""You are evaluating the clarity of a counter-narrative message
intended for the general public (8th grade reading level).

Rate the following text on a scale of 1 to 5:
1 = Very unclear, confusing, jargon-heavy
2 = Somewhat unclear, hard to follow
3 = Moderately clear, understandable with effort
4 = Clear, easy to understand
5 = Very clear, immediately understandable

You MUST respond ONLY with a valid JSON object like this:
{{"clarity_score": 4, "reason": "The text is straightforward and easy to follow."}}

Do NOT include markdown, backticks, or any text outside the JSON.

Text to evaluate:
\"\"\"{text[:1000]}\"\"\"
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        raw = response.choices[0].message.content.strip()


        # Strip markdown ticks
        cleaned = re.sub(r"```(?:json)?\s*", "", raw).replace("```", "").strip()

        try:
            parsed = json.loads(cleaned)
        except:
            # Extract first JSON block
            match = re.search(r"\{.*?\}", cleaned, re.DOTALL)
            if match:
                parsed = json.loads(match.group())
            else:
                print(f"PARSE FAILED. Raw: {raw[:200]}")
                return -1, raw[:200]

        score = int(parsed.get("clarity_score", -1))
        reason = parsed.get("reason", "")
        return score, reason

    except Exception as e:
        print(f"API ERROR: {str(e)[:300]}")
        return -1, str(e)

In [ ]:
import re
import json
print("\n GPT-4.1-Mini CLARITY SCORES") # GPT-4o-Mini was not working, the model was changed as it was advised by the TA

for model_name in all_models:
    for persona_name in persona_fns:
        col = f"{model_name}_{persona_name}"

        if col not in counter_df.columns:
            print(f"{col}: column not found")
            continue

        scores = []
        errors = 0

        for i, text in enumerate(counter_df[col].tolist()):
            score, reason = gpt_clarity_score(text, debug=(i == 0))  # debug first row only
            if score != -1:
                scores.append(score)
            else:
                errors += 1

        if scores:
            print(f"{col}: avg clarity = {sum(scores)/len(scores):.2f} | valid={len(scores)} | errors={errors}")
        else:
            print(f"{col}: NO valid scores — all {errors} failed")


 GPT-4o-Mini CLARITY SCORES
gpt4_mini_persona_0_vanilla: avg clarity = 4.90 | valid=87 | errors=13
gpt4_mini_persona_1_educator: avg clarity = 4.78 | valid=93 | errors=7
gpt4_mini_persona_2_ngo: avg clarity = 4.98 | valid=94 | errors=6
gpt4_mini_persona_3_law_enforcement: avg clarity = 4.53 | valid=92 | errors=8
llama_persona_0_vanilla: avg clarity = 4.89 | valid=100 | errors=0
llama_persona_1_educator: avg clarity = 4.51 | valid=100 | errors=0
llama_persona_2_ngo: avg clarity = 4.89 | valid=100 | errors=0
llama_persona_3_law_enforcement: avg clarity = 4.68 | valid=100 | errors=0
claude_sonnet_persona_0_vanilla: avg clarity = 4.19 | valid=100 | errors=0
claude_sonnet_persona_1_educator: avg clarity = 3.86 | valid=100 | errors=0
claude_sonnet_persona_2_ngo: avg clarity = 4.11 | valid=100 | errors=0
claude_sonnet_persona_3_law_enforcement: avg clarity = 3.81 | valid=100 | errors=0
claude_haiku_persona_0_vanilla: avg clarity = 4.25 | valid=100 | errors=0
claude_haiku_persona_1_educator: 

## LLM Clarity Analysis
The LLM clarity scores are consistently high across all models and personas, with most averages ranging between 4.5 and 5. This suggests that the generated counter-narratives are generally clear and easy to understand for a general audience.

NGO and vanilla personas tend to have the highest clarity scores, while law enforcement responses score slightly lower due to the use of legal and formal language.

In [ ]:
# Pick one column to analyze disagreements
col = "gpt4_mini_persona_1_educator"

disagreements = []
for _, row in counter_df.iterrows():
    text = row[col]
    if not isinstance(text, str) or len(text.strip()) == 0:
        continue

    flesch = textstat.flesch_reading_ease(text)
    flesch_scaled = flesch_to_scale(flesch)
    llm_score, reason = gpt_clarity_score(text)

    if llm_score != -1 and abs(llm_score - flesch_scaled) >= 2:
        disagreements.append({
            "text": text[:200],
            "flesch_raw": round(flesch, 2),
            "flesch_scaled": flesch_scaled,
            "llm_clarity": llm_score,
            "llm_reason": reason,
            "gap": abs(llm_score - flesch_scaled)
        })

    if len(disagreements) == 5:
        break

disagreement_df = pd.DataFrame(disagreements)
print(disagreement_df[["flesch_scaled", "llm_clarity", "gap", "llm_reason"]])

   flesch_scaled  llm_clarity  gap  \
0              1            5    4   
1              1            5    4   
2              1            5    4   
3              1            5    4   
4              1            5    4   

                                                                                              llm_reason  
0  The text uses simple language and clear sentences that are easy to understand for the general public.  
1  The text uses simple language and clear sentences that are easy to understand for the general public.  
2   The text is straightforward, uses simple language, and conveys its message clearly and respectfully.  
3    The text is straightforward, uses simple language, and conveys its message clearly and effectively.  
4   The text uses simple language and clear concepts that are easy to understand for the general public.  


## Comparison Analysis: LLM vs Readability
There is a strong disagreement between LLM clarity scores and traditional readability metrics. In multiple examples, the readability score is very low (scaled score = 1), while the LLM clarity score is very high (score = 5), resulting in a large gap of 4 points.

This disagreement occurs because readability metrics rely only on sentence length and word complexity, while LLM evaluation considers meaning, structure, and overall understanding. For example, texts with longer sentences and formal vocabulary are penalized by readability formulas but are still rated as clear by the LLM because they are logically structured and coherent.

Therefore, LLM-based evaluation is more accurate for counter-narratives, as it better reflects how real people understand the message rather than just measuring surface-level text features.